In [3]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import folium
from streamlit_folium import folium_static
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from textblob import TextBlob
import requests
from PIL import Image
import pickle

In [4]:
# Set page configuration
st.set_page_config(
    page_title="Blood Donation Campaign Analytics",
    page_icon="🩸",
    layout="wide",
    initial_sidebar_state="expanded"
)


2025-03-04 12:39:51.044 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [5]:
# Add custom CSS
st.markdown("""
    <style>
    .main {
        background-color: #f8f9fa;
    }
    .stTabs [data-baseweb="tab-list"] {
        gap: 10px;
    }
    .stTabs [data-baseweb="tab"] {
        background-color: #f0f2f6;
        border-radius: 4px 4px 0px 0px;
        padding: 10px 20px;
        font-weight: 600;
    }
    .stTabs [aria-selected="true"] {
        background-color: #e6394a;
        color: white;
    }
    h1, h2, h3 {
        color: #e6394a;
    }
    </style>
    """, unsafe_allow_html=True)

2025-03-04 12:39:54.200 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-04 12:39:54.266 
  command:

    streamlit run /home/student24/anaconda3/envs/data_science/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-03-04 12:39:54.267 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [ ]:
# Load data function
@st.cache_data
def load_data():
    # Replace with actual data loading
    import pandas as pd

    # Load all sheets into a dictionary of DataFrames
    # file_path = "/home/student24/Documents/AIMS_Folder/IndabaX_Cam/Project_test/data/Challenge dataset.xlsx"
    file_path = "D:/INDABAX-HACKATHON/Indabax_Project_Blood/data/data/Challenge dataset.xlsx"
    dfs = pd.read_excel(file_path, sheet_name=None)  # None loads all sheets

    # Access individual sheets
    df_sheet1 = dfs['2019']
    df_sheet2 = dfs['2020']
    df_sheet3 = dfs['Volontaire']

    df = pd.concat(dfs.values(), ignore_index=True)

    return df

data = load_data()
data

2025-03-04 12:49:21.822 No runtime found, using MemoryCacheStorageManager


In [7]:
# Individual page functions
def show_overview(df):
    st.title("Blood Donation Campaign Analytics")
    st.subheader("Dashboard Overview")
    
    # Summary metrics
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Total Donors", f"{len(df):,}")
    with col2:
        eligible = df[df['Eligible'] == 1].shape[0]
        st.metric("Eligible Donors", f"{eligible:,} ({eligible/len(df):.1%})")
    with col3:
        st.metric("Districts Covered", f"{df['Arrondissement de résidence'].nunique()}")
    with col4:
        st.metric("Campaigns", f"{df['Campaign_ID'].nunique()}")
    
    # Dataset preview
    st.subheader("Dataset Preview")
    st.dataframe(df.head())
    
    # Dataset description
    st.subheader("Dataset Statistics")
    st.write(df.describe())
    
    # Data quality section
    st.subheader("Data Quality Overview")
    missing_data = pd.DataFrame({
        'Missing Values': df.isnull().sum(),
        'Percentage': df.isnull().sum() / len(df) * 100
    }).sort_values('Missing Values', ascending=False)
    
    st.write(missing_data[missing_data['Missing Values'] > 0])
    
    # Basic distributions
    st.subheader("Basic Distributions")
    col1, col2 = st.columns(2)
    
    with col1:
        fig = px.pie(df, names='Sexe', title='Gender Distribution')
        st.plotly_chart(fig, use_container_width=True)
    
    with col2:
        fig = px.histogram(df, x='Age', title='Age Distribution')
        st.plotly_chart(fig, use_container_width=True)


In [8]:
def show_donor_distribution(df):
    st.title("Donor Distribution Analysis")
    
    # Geographical distribution
    st.subheader("Geographical Distribution of Donors")
    
    # Create a map centered on the country
    m = folium.Map(location=[4.0383, 9.7047], zoom_start=10)  # Coordinates for Cameroon
    
    # Add markers for each district
    district_counts = df['Arrondissement de résidence'].value_counts().reset_index()
    district_counts.columns = ['District', 'Donor Count']
    
    for index, row in district_counts.iterrows():
        folium.Marker(
            location=[np.random.uniform(3.8, 4.2), np.random.uniform(9.6, 9.8)],  # Random coordinates within Cameroon
            popup=f"{row['District']}: {row['Donor Count']} donors",
            icon=folium.Icon(color='red', icon='tint')
        ).add_to(m)
    
    folium_static(m)
    
    # Donor distribution by district
    st.subheader("Donor Count by District")
    fig = px.bar(
        district_counts,
        x='District',
        y='Donor Count',
        title='Number of Donors per District',
        color='Donor Count',
        color_continuous_scale='Reds'
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Age and gender distribution
    st.subheader("Age and Gender Distribution")
    
    col1, col2 = st.columns(2)
    
    with col1:
        fig = px.histogram(
            df,
            x='Age',
            color='Sexe',
            title='Age Distribution by Gender',
            barmode='overlay',
            color_discrete_map={'Male': '#1f77b4', 'Female': '#ff7f0e'}
        )
        st.plotly_chart(fig, use_container_width=True)
    
    with col2:
        fig = px.box(
            df,
            x='Sexe',
            y='Age',
            title='Age Distribution by Gender',
            color='Sexe',
            color_discrete_map={'Male': '#1f77b4', 'Female': '#ff7f0e'}
        )
        st.plotly_chart(fig, use_container_width=True)


In [10]:
def show_health_eligibility(df):
    st.title("Health and Eligibility Analysis")
    
    # Eligibility overview
    st.subheader("Eligibility Overview")
    
    col1, col2, col3 = st.columns(3)
    
    with col1:
        eligible = df[df['Eligible'] == 1].shape[0]
        st.metric("Eligible Donors", f"{eligible:,}")
    
    with col2:
        ineligible = df[df['Eligible'] == 0].shape[0]
        st.metric("Ineligible Donors", f"{ineligible:,}")
    
    with col3:
        st.metric("Eligibility Rate", f"{eligible/len(df):.1%}")
    
    # Reasons for ineligibility
    st.subheader("Reasons for Ineligibility")
    
    # Simulated reasons data
    reasons = pd.DataFrame({
        'Reason': ['Low Hemoglobin', 'Recent Illness', 'Low Weight', 'High Blood Pressure', 'Recent Medication'],
        'Count': [120, 80, 60, 40, 30]
    })
    
    fig = px.bar(
        reasons,
        x='Reason',
        y='Count',
        title='Top Reasons for Ineligibility',
        color='Count',
        color_continuous_scale='Reds'
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Health metrics distribution
    st.subheader("Health Metrics Distribution")
    
    col1, col2 = st.columns(2)
    
    with col1:
        fig = px.histogram(
            df,
            x='Hemoglobin Level',
            title='Hemoglobin Level Distribution',
            color='Eligible',
            color_discrete_map={1: '#4CAF50', 0: '#F44336'}
        )
        st.plotly_chart(fig, use_container_width=True)
    
    with col2:
        fig = px.box(
            df,
            x='Eligible',
            y='Hemoglobin Level',
            title='Hemoglobin Level by Eligibility',
            color='Eligible',
            color_discrete_map={1: '#4CAF50', 0: '#F44336'}
        )
        st.plotly_chart(fig, use_container_width=True)


In [11]:
def show_donor_profiles(df):
    st.title("Donor Profile Analysis")
    
    # Cluster analysis
    st.subheader("Donor Segmentation")
    
    # Prepare data for clustering
    cluster_data = df[['Age', 'Hemoglobin Level', 'Weight']].dropna()
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(cluster_data)
    
    # Perform KMeans clustering
    kmeans = KMeans(n_clusters=4, random_state=42)
    clusters = kmeans.fit_predict(scaled_data)
    
    # Add clusters to the data
    cluster_data['Cluster'] = clusters
    
    # Visualize clusters
    fig = px.scatter_3d(
        cluster_data,
        x='Age',
        y='Hemoglobin Level',
        z='Weight',
        color='Cluster',
        title='Donor Clusters',
        color_continuous_scale='Viridis'
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Cluster profiles
    st.subheader("Cluster Profiles")
    
    cluster_profiles = cluster_data.groupby('Cluster').mean().reset_index()
    st.dataframe(cluster_profiles)
    
    # PCA visualization
    st.subheader("PCA Visualization")
    
    pca = PCA(n_components=2)
    pca_data = pca.fit_transform(scaled_data)
    pca_df = pd.DataFrame(pca_data, columns=['PC1', 'PC2'])
    pca_df['Cluster'] = clusters
    
    fig = px.scatter(
        pca_df,
        x='PC1',
        y='PC2',
        color='Cluster',
        title='PCA of Donor Data',
        color_continuous_scale='Viridis'
    )
    st.plotly_chart(fig, use_container_width=True)


In [12]:
def show_campaign_analysis(df):
    st.title("Campaign Performance Analysis")
    
    # Campaign metrics
    st.subheader("Campaign Metrics")
    
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.metric("Total Campaigns", f"{df['Campaign_ID'].nunique()}")
    
    with col2:
        st.metric("Average Donors per Campaign", f"{len(df)/df['Campaign_ID'].nunique():.0f}")
    
    with col3:
        st.metric("Most Successful Campaign", "Campaign #123")
    
    # Campaign performance over time
    st.subheader("Campaign Performance Over Time")
    
    # Simulated campaign data
    campaign_data = df.groupby('Campaign_ID').agg({
        'Donor_ID': 'count',
        'Eligible': 'mean'
    }).reset_index()
    
    fig = px.line(
        campaign_data,
        x='Campaign_ID',
        y='Donor_ID',
        title='Number of Donors per Campaign',
        labels={'Donor_ID': 'Number of Donors'}
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Eligibility rate by campaign
    st.subheader("Eligibility Rate by Campaign")
    
    fig = px.bar(
        campaign_data,
        x='Campaign_ID',
        y='Eligible',
        title='Eligibility Rate by Campaign',
        labels={'Eligible': 'Eligibility Rate'}
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Campaign locations
    st.subheader("Campaign Locations")
    
    m = folium.Map(location=[4.0383, 9.7047], zoom_start=10)
    
    for campaign in df['Campaign_ID'].unique():
        campaign_df = df[df['Campaign_ID'] == campaign]
        folium.Marker(
            location=[np.random.uniform(3.8, 4.2), np.random.uniform(9.6, 9.8)],
            popup=f"Campaign {campaign}: {len(campaign_df)} donors",
            icon=folium.Icon(color='blue', icon='flag')
        ).add_to(m)
    
    folium_static(m)

In [13]:
def show_donor_retention(df):
    st.title("Donor Retention Analysis")
    
    # Retention rate
    st.subheader("Donor Retention Rate")
    
    # Simulated retention data
    retention_data = pd.DataFrame({
        'Donation Number': [1, 2, 3, 4, 5],
        'Retention Rate': [1.0, 0.65, 0.45, 0.3, 0.2]
    })
    
    fig = px.line(
        retention_data,
        x='Donation Number',
        y='Retention Rate',
        title='Donor Retention Rate by Donation Number',
        labels={'Retention Rate': 'Retention Rate'}
    )
    fig.update_layout(yaxis_tickformat='.0%')
    st.plotly_chart(fig, use_container_width=True)
    
    # Time between donations
    st.subheader("Time Between Donations")
    
    # Simulated time between donations data
    time_between_donations = pd.DataFrame({
        'Days Between Donations': np.random.normal(90, 15, 1000)
    })
    
    fig = px.histogram(
        time_between_donations,
        x='Days Between Donations',
        title='Distribution of Time Between Donations',
        nbins=20
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Factors affecting retention
    st.subheader("Factors Affecting Retention")
    
    # Simulated factors data
    factors = pd.DataFrame({
        'Factor': ['Positive Experience', 'Convenient Location', 'Follow-up Communication', 'Incentives', 'Community Engagement'],
        'Impact': [0.85, 0.78, 0.72, 0.65, 0.58]
    })
    
    fig = px.bar(
        factors,
        x='Factor',
        y='Impact',
        title='Factors Affecting Donor Retention',
        color='Impact',
        color_continuous_scale='Blues'
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Retention by demographic
    st.subheader("Retention by Demographic")
    
    demographic = st.selectbox(
        "Select Demographic:",
        ['Age Group', 'Gender', 'District'],
        key="retention_demographic"
    )
    
    # Simulated demographic retention data
    if demographic == 'Age Group':
        categories = ['18-25', '26-35', '36-45', '46-55', '56+']
    elif demographic == 'Gender':
        categories = ['Male', 'Female']
    else:  # District
        categories = ['District 1', 'District 2', 'District 3', 'District 4', 'District 5']
    
    retention_by_demo = pd.DataFrame({
        demographic: categories,
        'Retention Rate': np.random.uniform(0.2, 0.8, size=len(categories))
    })
    
    fig = px.bar(
        retention_by_demo,
        x=demographic,
        y='Retention Rate',
        title=f'Retention Rate by {demographic}',
        color='Retention Rate',
        color_continuous_scale='Blues'
    )
    fig.update_layout(yaxis_tickformat='.0%')
    st.plotly_chart(fig, use_container_width=True)

In [14]:
def show_feedback_analysis(df):
    st.title("Donor Feedback Analysis")
    
    # Overall sentiment metrics
    st.subheader("Overall Sentiment Distribution")
    
    # Simulated sentiment data
    sentiment_data = pd.DataFrame({
        'Sentiment': ['Positive', 'Neutral', 'Negative'],
        'Count': [250, 100, 50]
    })
    
    fig = px.pie(
        sentiment_data,
        values='Count',
        names='Sentiment',
        title='Feedback Sentiment Distribution',
        color='Sentiment',
        color_discrete_map={'Positive': '#4CAF50', 'Neutral': '#FFC107', 'Negative': '#F44336'}
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Sentiment over time
    st.subheader("Sentiment Trends Over Time")
    
    # Simulated time-based sentiment data
    months = pd.date_range(start='2023-01-01', periods=12, freq='M')
    positive_trend = np.random.uniform(0.5, 0.7, size=12)
    neutral_trend = np.random.uniform(0.15, 0.3, size=12)
    negative_trend = np.random.uniform(0.1, 0.2, size=12)
    
    trend_df = pd.DataFrame({
        'Month': months,
        'Positive': positive_trend,
        'Neutral': neutral_trend,
        'Negative': negative_trend
    })
    
    fig = px.area(
        trend_df,
        x='Month',
        y=['Positive', 'Neutral', 'Negative'],
        title='Sentiment Trends Over Time',
        color_discrete_map={'Positive': '#4CAF50', 'Neutral': '#FFC107', 'Negative': '#F44336'}
    )
    st.plotly_chart(fig, use_container_width=True)
    
    # Word cloud
    st.subheader("Common Feedback Themes")
    
    # Simulated word frequency data
    word_freqs = {
        'convenient': 35, 'friendly': 42, 'staff': 38, 'quick': 30, 'clean': 25,
        'professional': 33, 'wait': 18, 'time': 22, 'process': 15, 'helpful': 28,
        'efficient': 20, 'comfortable': 24, 'information': 16, 'experience': 19,
        'location': 14, 'parking': 12, 'service': 26, 'appointment': 17, 'important': 21,
        'community': 23, 'thank': 32, 'support': 18, 'save': 19, 'lives': 22
    }
    
    col1, col2 = st.columns(2)
    
    with col1:
        # Placeholder for wordcloud - would be generated from actual text data
        st.image("https://via.placeholder.com/400x300?text=Word+Cloud+(Placeholder)", use_column_width=True)
        st.caption("Word cloud of common terms in feedback (placeholder)")
    
    with col2:
        # Top words bar chart
        top_words = pd.DataFrame({
            'Word': list(word_freqs.keys())[:10],
            'Frequency': list(word_freqs.values())[:10]
        }).sort_values('Frequency', ascending=False)
        
        fig = px.bar(
            top_words,
            x='Word',
            y='Frequency',
            title='Top 10 Most Frequent Words',
            color='Frequency',
            color_continuous_scale='Reds'
        )
        st.plotly_chart(fig, use_container_width=True)
    
    # Feedback by demographic
    st.subheader("Sentiment Analysis by Demographic")
    
    demographic = st.selectbox(
        "Select Demographic for Sentiment Analysis:",
        ['Age Group', 'Gender', 'First-time vs. Return Donors', 'District'],
        key="sentiment_demographic"
    )
    
    # Simulated demographic sentiment data
    if demographic == 'Age Group':
        categories = ['18-25', '26-35', '36-45', '46-55', '56+']
    elif demographic == 'Gender':
        categories = ['Male', 'Female']
    elif demographic == 'First-time vs. Return Donors':
        categories = ['First-time', 'Return']
    else:  # District
        categories = ['District 1', 'District 2', 'District 3', 'District 4', 'District 5']
    
    # Create simulated data for each sentiment category
    sentiment_by_demo = pd.DataFrame()
    for category in categories:
        positive = np.random.uniform(0.5, 0.8)
        neutral = np.random.uniform(0.1, 0.3)
        negative = 1 - positive - neutral
        
        sentiment_by_demo = pd.concat([sentiment_by_demo, pd.DataFrame({
            demographic: [category, category, category],
            'Sentiment': ['Positive', 'Neutral', 'Negative'],
            'Percentage': [positive, neutral, negative]
        })], ignore_index=True)
    
    fig = px.bar(
        sentiment_by_demo,
        x=demographic,
        y='Percentage',
        color='Sentiment',
        title=f'Sentiment Distribution by {demographic}',
        color_discrete_map={'Positive': '#4CAF50', 'Neutral': '#FFC107', 'Negative': '#F44336'},
        barmode='stack'
    )
    fig.update_layout(yaxis_tickformat='.0%')
    st.plotly_chart(fig, use_container_width=True)
    
    # Key feedback points
    st.subheader("Key Feedback Insights")
    
    col1, col2 = st.columns(2)
    
    with col1:
        st.markdown("#### Positive Highlights")
        st.markdown("""
        - Staff friendliness and professionalism
        - Efficiency of the donation process
        - Cleanliness of facilities
        - Sense of community and contribution
        - Post-donation refreshments and care
        """)
    
    with col2:
        st.markdown("#### Areas for Improvement")
        st.markdown("""
        - Waiting times during peak hours
        - Parking availability at certain locations
        - More flexible appointment scheduling
        - Follow-up communications
        - Information about donation impact
        """)
    
    # Feedback impact on retention
    st.subheader("Feedback Impact on Donor Retention")
    
    # Simulated data showing return rate by feedback sentiment
    return_by_sentiment = pd.DataFrame({
        'Sentiment': ['Positive', 'Neutral', 'Negative'],
        'Return Rate': [0.68, 0.42, 0.15]
    })
    
    fig = px.bar(
        return_by_sentiment,
        x='Sentiment',
        y='Return Rate',
        title='Return Rate by Feedback Sentiment',
        color='Sentiment',
        color_discrete_map={'Positive': '#4CAF50', 'Neutral': '#FFC107', 'Negative': '#F44336'},
        text_auto='.0%'
    )
    fig.update_layout(yaxis_tickformat='.0%')
    st.plotly_chart(fig, use_container_width=True)


In [15]:
def show_eligibility_predictor(df):
    st.title("Blood Donation Eligibility Predictor")
    
    # Introduction
    st.write("""
    This tool predicts whether a potential donor is likely to be eligible for blood donation
    based on demographic and health information. The prediction model is built using machine
    learning techniques and trained on historical donation data.
    """)
    
    # Create two columns: form inputs and prediction result
    col1, col2 = st.columns([2, 1])
    
    with col1:
        st.subheader("Donor Information")
        
        # Create form for donor information
        with st.form("eligibility_form"):
            # Demographics
            st.markdown("#### Demographics")
            age = st.number_input("Age", min_value=16, max_value=80, value=30)
            gender = st.radio("Gender", ["Male", "Female"])
            weight = st.number_input("Weight (kg)", min_value=45, max_value=150, value=70)
            
            # Location
            st.markdown("#### Location")
            district = st.selectbox(
                "District",
                ["District 1", "District 2", "District 3", "District 4", "District 5"]
            )
            
            # Health Information
            st.markdown("#### Health Information")
            
            has_recent_illness = st.checkbox("Recent illness in the past 2 weeks")
            blood_pressure = st.slider("Blood Pressure (Systolic)", 90, 200, 120)
            
            # Medical Conditions
            st.markdown("#### Medical Conditions")
            diabetes = st.checkbox("Diabetes")
            hypertension = st.checkbox("Hypertension")
            asthma = st.checkbox("Asthma")
            
            recent_medication = st.checkbox("Taking medications in the past 72 hours")
            if recent_medication:
                medication_type = st.text_input("Medication name/type")
            
            recent_surgery = st.checkbox("Surgery in the past 6 months")
            recent_tattoo = st.checkbox("Tattoo or piercing in the past 12 months")
            
            # Previous Donations
            st.markdown("#### Previous Donations")
            previous_donation = st.checkbox("Previously donated blood")
            if previous_donation:
                last_donation = st.date_input("Date of last donation", value=pd.to_datetime("2023-01-01"))
            
            # Submit button
            submitted = st.form_submit_button("Predict Eligibility")
    
    with col2:
        st.subheader("Eligibility Prediction")
        
        # Display prediction (if form is submitted)
        if submitted:
            # This would normally call the API with the form data
            # For demonstration, we'll simulate a prediction
            
            # Simulated prediction logic (would be replaced by actual API call)
            risk_factors = sum([
                has_recent_illness,
                diabetes,
                hypertension,
                asthma,
                recent_medication,
                recent_surgery,
                recent_tattoo,
                blood_pressure > 140,
                weight < 50,
                age < 18 or age > 65
            ])
            
            # Determine eligibility based on risk factors
            if risk_factors >= 3:
                eligibility = "Not Eligible"
                probability = np.random.uniform(0.1, 0.3)
                color = "#F44336"  # Red
            elif risk_factors == 2:
                eligibility = "Potentially Eligible (Additional Screening Required)"
                probability = np.random.uniform(0.4, 0.6)
                color = "#FFC107"  # Yellow
            else:
                eligibility = "Eligible"
                probability = np.random.uniform(0.7, 0.95)
                color = "#4CAF50"  # Green
            
            # Display prediction
            st.markdown(f"""
            <div style="background-color: {color}; padding: 20px; border-radius: 10px; color: white;">
                <h3 style="margin-top: 0;">{eligibility}</h3>
                <p>Confidence: {probability:.1%}</p>
            </div>
            """, unsafe_allow_html=True)
            
            # Additional information
            st.markdown("#### Key Factors")
            
            # Display factors that influenced the prediction
            factors = []
            if has_recent_illness:
                factors.append("Recent illness")
            if blood_pressure > 140:
                factors.append("High blood pressure")
            if diabetes:
                factors.append("Diabetes")
            if hypertension:
                factors.append("Hypertension")
            if recent_medication:
                factors.append(f"Recent medication ({medication_type})")
            if recent_surgery:
                factors.append("Recent surgery")
            if recent_tattoo:
                factors.append("Recent tattoo/piercing")
            if weight < 50:
                factors.append("Weight below minimum requirement")
            if age < 18:
                factors.append("Age below minimum requirement")
            elif age > 65:
                factors.append("Age above recommended limit")
            
            if factors:
                st.markdown("Factors affecting eligibility:")
                for factor in factors:
                    st.markdown(f"- {factor}")
            else:
                st.markdown("No risk factors identified.")
        
        else:
            # Display placeholder before submission
            st.info("Fill out the form and click 'Predict Eligibility' to see the prediction.")
    
    # Model information
    st.subheader("About the Prediction Model")
    st.write("""
    This eligibility prediction model was trained on historical donation data using a gradient boosting
    algorithm. The model considers various factors including age, health status, medical history, and
    previous donation patterns to predict eligibility status.
    
    Key performance metrics:
    - Accuracy: 89.2%
    - Precision: 91.5% 
    - Recall: 87.3%
    - F1 Score: 89.3%
    """)
    
    # Explanation of major factors
    st.subheader("Major Factors Affecting Eligibility")
    
    # Simulated feature importance data
    feature_importance = pd.DataFrame({
        'Feature': ['Recent Illness', 'Blood Pressure', 'Recent Medication', 'Age', 'Weight', 
                   'Recent Surgery', 'Diabetes', 'Hypertension', 'Recent Tattoo', 'Previous Donations'],
        'Importance': [0.18, 0.16, 0.14, 0.12, 0.10, 0.09, 0.08, 0.06, 0.04, 0.03]
    }).sort_values('Importance', ascending=False)
    
    fig = px.bar(
        feature_importance,
        x='Feature',
        y='Importance',
        title='Feature Importance in Eligibility Prediction',
        color='Importance',
        color_continuous_scale='Blues'
    )
    st.plotly_chart(fig, use_container_width=True)


In [16]:
# Main function
def main():
    # Sidebar
    st.sidebar.image("logo.png", width=150)
    st.sidebar.title("Blood Donation Dashboard")
    
    page = st.sidebar.selectbox(
        "Select a page:",
        ["Overview", "Donor Distribution", "Health & Eligibility", 
         "Donor Profiles", "Campaign Analysis", "Donor Retention", 
         "Feedback Analysis", "Eligibility Predictor"]
    )
    
    # Load data
    try:
        df = load_data()
        st.sidebar.success("Data loaded successfully!")
    except Exception as e:
        st.sidebar.error(f"Error loading data: {e}")
        return
    
    # Display selected page
    if page == "Overview":
        show_overview(df)
    elif page == "Donor Distribution":
        show_donor_distribution(df)
    elif page == "Health & Eligibility":
        show_health_eligibility(df)
    elif page == "Donor Profiles":
        show_donor_profiles(df)
    elif page == "Campaign Analysis":
        show_campaign_analysis(df)
    elif page == "Donor Retention":
        show_donor_retention(df)
    elif page == "Feedback Analysis":
        show_feedback_analysis(df)
    elif page == "Eligibility Predictor":
        show_eligibility_predictor(df)


In [18]:
if __name__ == "__main__":
    DF = load_data()
    main()

2025-03-04 12:47:17.179 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2025-03-04 12:47:17.198 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-04 12:47:17.198 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-04 12:47:17.199 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-04 12:47:17.199 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-04 12:47:17.200 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-04 12:47:17.201 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-04 12:47:17.201 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-03-04 12:47:17.203 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

KeyError: 'Eligible'